# cProfiles for Mandelbrot Set Implementations

## Imports

In [1]:
from pathlib import Path
import cProfile, pstats
import numpy as np
from Naive.mandelbrot_grid import compute_mandelbrot_grid as mandelbrot_naive
from Numpy_Vectorization.mandelbrot_mesh import compute_mandelbrot_grid as mandelbrot_numpy

In [2]:
# regions
x_min, x_max = -2, 1
y_min, y_max = -1.5, 1.5

res = 1024

x_region = np.linspace(x_min, x_max, res)
y_region = np.linspace(y_min, y_max, res)

max_iterations = 100
bound = 2
power = 2

# parent folder
pf = 'Profilers/cProfiles'

# naive
cProfile.run('mandelbrot_naive(x_region, y_region, max_iterations, bound, power, res)', 
                f'{pf}/naive_profile.prof')
# numpy vectorized
cProfile.run('mandelbrot_numpy(x_region, y_region, max_iterations, bound, power, res)', 
                f'{pf}/numpy_profile.prof')

for name in (f'{pf}/naive_profile.prof', f'{pf}/numpy_profile.prof'):
    stats = pstats.Stats(name)
    stats.sort_stats('cumulative')
    stats.print_stats(5) # prints top 5
    
    # save stats to .txt
    profile = Path(name).stem
    output_path = f'{pf}/{profile}.txt'

    with open(output_path, "w") as f:
        stats.stream = f
        stats.print_stats() # stores all

Mon Mar  2 23:29:20 2026    Profilers/cProfiles/naive_profile.prof

         23878114 function calls in 6.305 seconds

   Ordered by: cumulative time
   List reduced from 6 to 5 due to restriction <5>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    6.305    6.305 {built-in method builtins.exec}
        1    0.003    0.003    6.305    6.305 <string>:1(<module>)
        1    5.027    5.027    6.303    6.303 c:\Python Projects\mandelbrot-nsc\Naive\mandelbrot_grid.py:5(compute_mandelbrot_grid)
 22828510    1.203    0.000    1.203    0.000 {built-in method builtins.abs}
  1049600    0.073    0.000    0.073    0.000 {method 'append' of 'list' objects}


Mon Mar  2 23:29:22 2026    Profilers/cProfiles/numpy_profile.prof

         51 function calls in 1.707 seconds

   Ordered by: cumulative time
   List reduced from 30 to 5 due to restriction <5>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000 

### Questions to Answer
1. Which functions takes the most time?
    - Naive: the most time is spend using `abs()` and `append()`
    - Numpy: the most time is spend updating the z-value
2. Are there functions called surprisingly many times?
    - Naive: `abs()` is called about 23 million times, which is the needed iteration count times the pixel-count. This also means that `append()` is called `res` x `res` amount of times
    - Numpy: only use the same amount of calls as `max_iteration`
3. How does Numpy profile compare to Naive?
    - Naive seems a little simpler, but Numpy does its computations much faster due to vectorization.
4. Where does Numpy spend its time?
    - It is a little difficult to say as it seems that most time spend is actually just passing in the parameters / calling the funcition...
5. How many functions appear in each profile? What does
this difference tell you about where the work actually happens?
    - Naive: very few function calls (10)
    - Numpy: many function calls (51)
    - The naive functions are integrated in the overall function, while for the Numpy implementation, most of the function calls are internal.

## Lineprofiler

In [ ]:
import os
import line_profiler

os.environ['LINE_PROFILE'] = '1' # makes sure that files are actually saved

# run this function using: 'kernprof -l -v mandelbrot_profiler.py' in the terminal
@line_profiler.profile
def compute_naive_mandelbrot(x_region, y_region, max_iterations, bound, power):
    mandelbrot_array = []

    for y_value in y_region:
        row = []

        for x_value in x_region:
            c = complex(x_value, y_value)
            z = 0

            for iteration in range(max_iterations):
                if(abs(z) >= bound):
                    row.append(iteration)
                    break
                else: 
                    z = z**power + c
            else: # this is only called if the for loop never 'breaks'
                row.append(max_iterations)
        mandelbrot_array.append(row)

    return mandelbrot_array

@line_profiler.profile
def compute_numpy_mandelbrot(x_mesh, y_mesh, max_iterations, bound, power):
    complex_number = 1j
    C = x_mesh + y_mesh * complex_number
    Z = np.zeros_like(C)
    M = np.zeros(C.shape, dtype=np.int32)

    for iteration in range(max_iterations):
        mask = np.abs(Z) <= bound
        Z[mask] = Z[mask]**power + C[mask]
        M[mask] += 1

    return M

# parameters
max_iterations = 100
bound = 2
power = 2
x_res, y_res = 1024, 1024

# regions
x_min, x_max = -2, 1
y_min, y_max = -1.5, 1.5
x_region = np.linspace(x_min, x_max, x_res)
y_region = np.linspace(y_min, y_max, y_res)
x_mesh, y_mesh = np.meshgrid(x_region, y_region)

# run the functions to get the line profiles
naive_mandelbrot = compute_naive_mandelbrot(x_region, y_region, max_iterations, bound, power)
numpy_mandelbrot = compute_numpy_mandelbrot(x_mesh, y_mesh, max_iterations, bound, power)


### Questions to Answer
1. Which lines dominate runtime? What fraction of total time is
spent in the inner loop?
    - For the Naive implementation, what dominates runtime is the failing if statement -> reassigning the z-value
    - Inner for loop(`24.2%`), if(`35.2%`), else(`33.4%`) = `92.8%` of the total time spent in the inner loop
2. Why is NumPy faster than naive Python?
    - The naive implementation checks every single pixel for up to `max_iterations` and reassigns the z-value, while the numpy implementation just does `max_iterations` amount of checks where it just updates the entire array at once. 
3. What would you need to change to make the naive version faster?
    - Well, since the inner loop takes up `92.8%` of the total runtime, we would probably want to optimize that. It is very inefficient to to reassign the z-value up to `max_iterations` times. We could probably even save compute time by just switching the order of the for loops, such that we only reassign the value of z a maximum of `max_iteration`.